# WP1 Step 1 — Build the frozen Gaia DR3 narrow catalogue

This notebook consolidates the five tiled Gaia DR3 downloads into the named WP1 artefact `wp1_gaia_narrow.cat`. It performs the exact local Galactic-coordinate cut, validates tile consistency and source uniqueness, and writes a machine-readable validation manifest.

**Upstream inputs:** `data/raw/gaia/wp1_gaia_narrow_tile01..05.parquet`

**Outputs:** `data/processed/wp1_gaia_narrow.parquet`, `data/processed/wp1_gaia_narrow.fits`, and `provenance/wp1_gaia_narrow_validation.json`

In [1]:
# Section 1 — Imports and reproducible paths
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from astropy.coordinates import SkyCoord
import astropy.units as u

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name != 'gaia_snr_history_cygnus':
    PROJECT_ROOT = Path('/Users/vdk/science/gaia_snr_history_cygnus')
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'gaia'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROVENANCE_DIR = PROJECT_ROOT / 'provenance'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PROVENANCE_DIR.mkdir(parents=True, exist_ok=True)
TILE_PATHS = sorted(RAW_DIR.glob('wp1_gaia_narrow_tile*.parquet'))
assert len(TILE_PATHS) == 5, f'Expected 5 narrow tiles, found {len(TILE_PATHS)}'
TILE_PATHS

[PosixPath('/Users/vdk/science/gaia_snr_history_cygnus/data/raw/gaia/wp1_gaia_narrow_tile01.parquet'),
 PosixPath('/Users/vdk/science/gaia_snr_history_cygnus/data/raw/gaia/wp1_gaia_narrow_tile02.parquet'),
 PosixPath('/Users/vdk/science/gaia_snr_history_cygnus/data/raw/gaia/wp1_gaia_narrow_tile03.parquet'),
 PosixPath('/Users/vdk/science/gaia_snr_history_cygnus/data/raw/gaia/wp1_gaia_narrow_tile04.parquet'),
 PosixPath('/Users/vdk/science/gaia_snr_history_cygnus/data/raw/gaia/wp1_gaia_narrow_tile05.parquet')]

In [2]:
# Section 2 — Read tiles and enforce a common schema
tile_tables = []
tile_summary = []
reference_columns = None
for tile_path in TILE_PATHS:
    table = pq.read_table(tile_path)
    columns = table.column_names
    if reference_columns is None:
        reference_columns = columns
    assert columns == reference_columns, f'Schema mismatch in {tile_path.name}'
    frame = table.to_pandas()
    frame['tile_id'] = tile_path.stem.rsplit('_', 1)[-1]
    tile_tables.append(frame)
    tile_summary.append({'file': str(tile_path.relative_to(PROJECT_ROOT)), 'rows': len(frame), 'sha256': hashlib.sha256(tile_path.read_bytes()).hexdigest()})

raw = pd.concat(tile_tables, ignore_index=True)
assert raw['source_id'].notna().all()
assert raw['source_id'].is_unique, 'Duplicate source_id found before coordinate filtering'
pd.DataFrame(tile_summary)

,file,rows,sha256
0,data/raw/gaia/wp1_gaia_narrow_tile01.parquet,178721,bb18e9ab92c6502b8147fe0727e465ddf08a5c77ac2642...
1,data/raw/gaia/wp1_gaia_narrow_tile02.parquet,212956,6aa8d0ef6dce81cbb8e810bc34b90d17aa0ec3c00c26f8...
2,data/raw/gaia/wp1_gaia_narrow_tile03.parquet,158843,1390da62206d4e045b9d8479032f8f03b6deb85519454a...
3,data/raw/gaia/wp1_gaia_narrow_tile04.parquet,185994,94b32a80bf427ac5cbdbd18d4d02ee0f99dac17e3277f0...
4,data/raw/gaia/wp1_gaia_narrow_tile05.parquet,113020,8a45d7122030ba5b03dea4b78161cdf2afb361eae90921...


In [3]:
# Section 3 — Apply the exact WP1 Galactic-coordinate selection locally
# The archive query deliberately used an enclosing ICRS cone; this is the authoritative cut.
coords = SkyCoord(ra=raw['ra'].to_numpy() * u.deg, dec=raw['dec'].to_numpy() * u.deg, frame='icrs')
galactic = coords.galactic
raw['l_deg'] = galactic.l.deg
raw['b_deg'] = galactic.b.deg

selection = (raw['l_deg'].between(77.0, 83.0, inclusive='both') &
             raw['b_deg'].between(-1.5, 4.0, inclusive='both'))
catalogue = raw.loc[selection].copy()
catalogue = catalogue.sort_values('source_id').reset_index(drop=True)
catalogue['local_selection'] = 'l[77,83], b[-1.5,4]'
print(f'Enclosing tiled rows: {len(raw):,}')
print(f'Exact narrow-catalogue rows: {len(catalogue):,}')
print(f'Rejected by local Galactic cut: {len(raw) - len(catalogue):,}')

Enclosing tiled rows: 849,534
Exact narrow-catalogue rows: 245,843
Rejected by local Galactic cut: 603,691


In [4]:
# Section 4 — Validation gate
required = ['source_id', 'ra', 'dec', 'l_deg', 'b_deg', 'parallax', 'parallax_error', 'phot_g_mean_mag', 'pmra', 'pmdec']
assert all(column in catalogue.columns for column in required)
assert catalogue['source_id'].is_unique
assert catalogue['l_deg'].between(77, 83).all()
assert catalogue['b_deg'].between(-1.5, 4).all()
assert catalogue['parallax'].between(0.35, 1.10).all()
assert (catalogue['phot_g_mean_mag'] < 19).all()

null_fraction = catalogue.isna().mean().sort_values(ascending=False).rename('null_fraction').to_frame()
quality_summary = pd.Series({
    'rows': len(catalogue),
    'unique_source_ids': catalogue['source_id'].nunique(),
    'duplicate_source_ids': int(catalogue['source_id'].duplicated().sum()),
    'rows_with_ruwe': int(catalogue['ruwe'].notna().sum()),
    'rows_with_radial_velocity': int(catalogue['radial_velocity'].notna().sum()),
    'rows_with_complete_bp_rp': int(catalogue[['phot_bp_mean_mag', 'phot_rp_mean_mag']].notna().all(axis=1).sum()),
})
quality_summary

rows                         245843
unique_source_ids            245843
duplicate_source_ids              0
rows_with_ruwe               245843
rows_with_radial_velocity     18242
rows_with_complete_bp_rp     242724
dtype: int64

In [5]:
# Section 5 — Write the frozen catalogue
# No parallax zero-point correction is applied here; WP2 will store raw, zero-point, and corrected values separately.
parquet_output = PROCESSED_DIR / 'wp1_gaia_narrow.parquet'
fits_output = PROCESSED_DIR / 'wp1_gaia_narrow.fits'
catalogue.to_parquet(parquet_output, index=False)
from astropy.table import Table
Table.from_pandas(catalogue, index=False).write(fits_output, overwrite=True)
print(parquet_output)
print(fits_output)

/Users/vdk/science/gaia_snr_history_cygnus/data/processed/wp1_gaia_narrow.parquet
/Users/vdk/science/gaia_snr_history_cygnus/data/processed/wp1_gaia_narrow.fits


In [6]:
# Section 6 — Write the audit manifest
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'notebook': 'notebooks/wp1_step1_build_narrow_catalogue.ipynb',
    'selection': {'l_deg': [77.0, 83.0], 'b_deg': [-1.5, 4.0], 'parallax_mas': [0.35, 1.10], 'phot_g_mean_mag_lt': 19.0},
    'input_tiles': tile_summary,
    'counts': {'input_rows': len(raw), 'output_rows': len(catalogue), 'rejected_by_local_galactic_cut': len(raw) - len(catalogue)},
    'validation': quality_summary.astype(int).to_dict(),
    'outputs': {str(path.relative_to(PROJECT_ROOT)): {'sha256': sha256(path), 'bytes': path.stat().st_size} for path in [parquet_output, fits_output]},
}
manifest_path = PROVENANCE_DIR / 'wp1_gaia_narrow_validation.json'
manifest_path.write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
print(manifest_path)
print(json.dumps(manifest['counts'], indent=2))

/Users/vdk/science/gaia_snr_history_cygnus/provenance/wp1_gaia_narrow_validation.json
{
  "input_rows": 849534,
  "output_rows": 245843,
  "rejected_by_local_galactic_cut": 603691
}
